# THU THẬP FEATURES CỦA BỆNH NHÂN
- Tổng hợp các features của bệnh nhân nằm ở các csv khác nhau, có khóa chung là subject_id, hadm_id, stay_id
- Khung thời gian thu thập là trong 12h đầu ở ICU
- Các nhóm đặc trưng chính: Nhân khẩu học, sinh hiệu, lượng nước tiểu,...
- Các đặc trưng time-series được rút gọn thành mean, median, min, max

### READ DATA

In [1]:
import pandas as pd
import numpy as np
import glob
import re
import os
os.chdir('/media/data3/users/tpp/mimic-derived/')

In [2]:
aki_df = pd.read_csv('/media/data3/users/tpp/MIMICIV_AKI_AMI/Thai/AKI_patients.csv', index_col=0)
aki_df['intime'] = pd.to_datetime(aki_df['intime'])
aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,prediction_start_time,prediction_end_time,aki_label
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,2129-08-05 00:45:00,2129-08-12 00:45:00,1
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,2141-05-23 08:18:01,2141-05-30 08:18:01,1
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,2187-02-24 04:02:25,2187-03-03 04:02:25,1
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,2132-12-15 21:29:01,2132-12-22 21:29:01,1
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,2167-11-08 08:22:00,2167-11-15 08:22:00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,2148-07-07 09:47:34,2148-07-14 09:47:34,1
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,2185-07-08 13:08:06,2185-07-15 13:08:06,1
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,2181-09-14 20:47:07,2181-09-21 20:47:07,1
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,2130-10-06 02:59:00,2130-10-13 02:59:00,1


### ICUSTAY_DETAIL

In [3]:
icustay_detail_df = pd.read_csv('demographics/icustay_detail.csv')

cols_to_use = [
    'subject_id', 'hadm_id', 'stay_id', 
    'race', 'hospstay_seq', 'first_hosp_stay'
]

aki_df = pd.merge(
    aki_df,
    icustay_detail_df[cols_to_use],
    on=['subject_id', 'hadm_id', 'stay_id'],
    how='left'
)

aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,prediction_start_time,prediction_end_time,aki_label,race,hospstay_seq,first_hosp_stay
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,2129-08-05 00:45:00,2129-08-12 00:45:00,1,WHITE,1,True
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,2141-05-23 08:18:01,2141-05-30 08:18:01,1,UNKNOWN,1,True
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,2187-02-24 04:02:25,2187-03-03 04:02:25,1,WHITE,1,True
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,2132-12-15 21:29:01,2132-12-22 21:29:01,1,WHITE,1,True
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,2167-11-08 08:22:00,2167-11-15 08:22:00,0,WHITE,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,2148-07-07 09:47:34,2148-07-14 09:47:34,1,WHITE,1,True
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,2185-07-08 13:08:06,2185-07-15 13:08:06,1,UNKNOWN,1,True
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,2181-09-14 20:47:07,2181-09-21 20:47:07,1,WHITE,1,True
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,2130-10-06 02:59:00,2130-10-13 02:59:00,1,WHITE,1,True


### WEIGHT

In [4]:
pd.read_csv('/media/data3/users/tpp/mimic-derived/firstday/first_day_weight.csv')

,subject_id,stay_id,weight_admit,weight,weight_min,weight_max
0,15174162,38182555,56.0,56.00,56.0,56.0
1,15674909,38223227,64.5,64.50,64.5,64.5
2,17155701,38398288,78.9,83.50,78.9,88.1
3,11282384,38614968,74.1,74.15,74.1,74.2
4,16248103,38868865,82.5,82.50,82.5,82.5
...,...,...,...,...,...,...
94453,17119162,32859560,NaN,NaN,NaN,NaN
94454,17446941,32865221,NaN,NaN,NaN,NaN
94455,17787227,30221516,NaN,NaN,NaN,NaN
94456,19232710,37056440,NaN,NaN,NaN,NaN


In [5]:
weight_df = pd.read_csv('/media/data3/users/tpp/mimic-derived/firstday/first_day_weight.csv')

# Lấy weight_admit từ first_day_weight
final_weights = weight_df[['subject_id', 'stay_id', 'weight_admit']].copy()
final_weights = final_weights.rename(columns={'weight_admit': 'weight'})

# Merge vào aki_df theo khóa chung (subject_id, stay_id)
aki_df = pd.merge(aki_df, final_weights, on=['subject_id', 'stay_id'], how='left')

print(f"Missing weight: {aki_df['weight'].isna().sum()} / {len(aki_df)} ({aki_df['weight'].isna().mean()*100:.2f}%)")
aki_df

Missing weight: 59 / 2316 (2.55%)


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,prediction_start_time,prediction_end_time,aki_label,race,hospstay_seq,first_hosp_stay,weight
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,2129-08-05 00:45:00,2129-08-12 00:45:00,1,WHITE,1,True,53.0
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,2141-05-23 08:18:01,2141-05-30 08:18:01,1,UNKNOWN,1,True,64.1
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,2187-02-24 04:02:25,2187-03-03 04:02:25,1,WHITE,1,True,87.7
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,2132-12-15 21:29:01,2132-12-22 21:29:01,1,WHITE,1,True,91.0
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,2167-11-08 08:22:00,2167-11-15 08:22:00,0,WHITE,1,True,79.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,2148-07-07 09:47:34,2148-07-14 09:47:34,1,WHITE,1,True,74.0
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,2185-07-08 13:08:06,2185-07-15 13:08:06,1,UNKNOWN,1,True,58.8
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,2181-09-14 20:47:07,2181-09-21 20:47:07,1,WHITE,1,True,113.9
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,2130-10-06 02:59:00,2130-10-13 02:59:00,1,WHITE,1,True,53.0


### HEIGHT

In [6]:
df_height = pd.read_csv('firstday/first_day_height.csv')

# Inner join vào aki_df theo subject_id và stay_id
aki_df = pd.merge(
    aki_df,
    df_height[['subject_id', 'stay_id', 'height']],
    on=['subject_id', 'stay_id'],
    how='left'
)

aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,prediction_start_time,prediction_end_time,aki_label,race,hospstay_seq,first_hosp_stay,weight,height
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,2129-08-05 00:45:00,2129-08-12 00:45:00,1,WHITE,1,True,53.0,NaN
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,2141-05-23 08:18:01,2141-05-30 08:18:01,1,UNKNOWN,1,True,64.1,170.0
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,2187-02-24 04:02:25,2187-03-03 04:02:25,1,WHITE,1,True,87.7,165.0
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,2132-12-15 21:29:01,2132-12-22 21:29:01,1,WHITE,1,True,91.0,173.0
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,2167-11-08 08:22:00,2167-11-15 08:22:00,0,WHITE,1,True,79.3,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,2148-07-07 09:47:34,2148-07-14 09:47:34,1,WHITE,1,True,74.0,157.0
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,2185-07-08 13:08:06,2185-07-15 13:08:06,1,UNKNOWN,1,True,58.8,NaN
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,2181-09-14 20:47:07,2181-09-21 20:47:07,1,WHITE,1,True,113.9,175.0
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,2130-10-06 02:59:00,2130-10-13 02:59:00,1,WHITE,1,True,53.0,NaN


### VITAL SIGN

In [7]:
pd.read_csv('measurement/vitalsign.csv')

,subject_id,stay_id,charttime,heart_rate,sbp,dbp,mbp,sbp_ni,dbp_ni,mbp_ni,resp_rate,temperature,temperature_site,spo2,glucose
0,10011938,36750867,2133-12-20 05:00:00,65.0,119.0,44.0,64.0,119.0,44.0,64.0,14.0,NaN,NaN,98.0,NaN
1,10011938,36750867,2133-12-21 16:00:00,69.0,132.0,49.0,73.0,132.0,49.0,73.0,16.0,36.67,Oral,95.0,NaN
2,10011938,36750867,2133-12-12 09:31:00,NaN,143.0,48.0,76.0,143.0,48.0,76.0,NaN,NaN,NaN,NaN,NaN
3,10011938,36750867,2133-12-12 16:01:00,NaN,129.0,49.0,73.0,129.0,49.0,73.0,6.0,NaN,NaN,NaN,NaN
4,10011938,36750867,2133-12-29 11:02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13519528,19989918,37503982,2176-10-25 13:00:00,62.0,136.0,91.0,101.0,136.0,91.0,101.0,25.0,NaN,NaN,95.0,NaN
13519529,19989918,30333293,2174-10-13 21:39:00,NaN,136.0,53.0,72.0,136.0,53.0,72.0,NaN,NaN,NaN,NaN,NaN
13519530,19989918,33220923,2178-07-15 13:00:00,58.0,112.0,58.0,72.0,112.0,58.0,72.0,18.0,NaN,NaN,97.0,NaN
13519531,19989918,33220923,2178-07-16 17:00:00,58.0,NaN,NaN,NaN,NaN,NaN,NaN,23.0,NaN,NaN,93.0,NaN


In [ ]:
df_vital = pd.read_csv('measurement/vitalsign.csv')
df_vital['charttime'] = pd.to_datetime(df_vital['charttime'])

merged_vital = pd.merge(
    aki_df[['stay_id', 'intime']],
    df_vital,
    on='stay_id',
    how='inner'
)

valid_vital = merged_vital[
    (merged_vital['charttime'] >= merged_vital['intime']) & 
    (merged_vital['charttime'] <= merged_vital['intime'] + pd.Timedelta(hours=12))
].copy()

# Lấy từ cột sbp trước, nếu không có thì lấy từ sbp_ni
valid_vital['sys_bp'] = valid_vital['sbp'].combine_first(valid_vital['sbp_ni'])
valid_vital['dias_bp'] = valid_vital['dbp'].combine_first(valid_vital['dbp_ni'])
valid_vital['mean_bp'] = valid_vital['mbp'].combine_first(valid_vital['mbp_ni'])

agg_dict = {
    'heart_rate': ['max'],
    'sys_bp':     ['min'],
    'mean_bp':    ['min'],
    'dias_bp':    ['min'],
    'resp_rate':  ['max'],
    'temperature':['max'],
    'spo2':       ['min'],
    'glucose':    ['max']
}

df_vital_features = valid_vital.groupby('stay_id').agg(agg_dict)
df_vital_features.columns = ['_'.join(col).strip() for col in df_vital_features.columns.values]
df_vital_features = df_vital_features.reset_index()

df_vital_features.columns = [col.replace('glucose_', 'glucose_vitalsign_') if 'glucose_' in col else col for col in df_vital_features.columns]

aki_df = pd.merge(aki_df, df_vital_features, on='stay_id', how='left')
aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,weight,height,heart_rate_max,sys_bp_min,mean_bp_min,dias_bp_min,resp_rate_max,temperature_max,spo2_min,glucose_vitalsign_max
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,53.0,NaN,83.0,98.0,66.0,46.0,20.0,36.61,89.0,95.0
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,64.1,170.0,115.0,72.0,47.0,40.0,26.0,37.06,94.0,331.0
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,87.7,165.0,93.0,100.0,77.0,43.0,52.0,37.61,97.0,93.0
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,91.0,173.0,100.0,87.0,51.0,35.0,25.0,37.30,93.0,214.0
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,79.3,NaN,74.0,101.0,65.0,56.0,24.0,36.89,97.0,111.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,74.0,157.0,89.0,90.0,51.0,37.0,23.0,36.94,93.0,166.0
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,58.8,NaN,78.0,93.0,54.0,38.0,23.0,36.78,92.0,177.0
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,113.9,175.0,100.0,97.0,73.0,59.0,26.0,37.33,96.0,203.0
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,53.0,NaN,76.0,110.0,68.0,53.0,24.0,36.83,95.0,174.0


### CHEMISTRY

In [9]:
pd.read_csv('measurement/chemistry.csv')

,subject_id,hadm_id,charttime,specimen_id,albumin,globulin,total_protein,aniongap,bicarbonate,bun,calcium,chloride,creatinine,glucose,sodium,potassium
0,10001823,NaN,2150-02-16 08:50:00,83686262,4.5,2.3,6.8,15.0,25.0,20.0,9.2,106.0,1.0,NaN,142.0,3.9
1,10001884,NaN,2123-04-28 12:40:00,10562194,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,92.0,NaN,NaN
2,10002557,NaN,2152-08-08 12:00:00,28354929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,112.0,NaN,NaN
3,10002557,NaN,2160-02-13 20:45:00,77330930,NaN,NaN,NaN,17.0,20.0,31.0,NaN,102.0,1.7,105.0,139.0,4.8
4,10004401,NaN,2141-05-02 14:35:00,80236524,NaN,NaN,NaN,15.0,26.0,22.0,9.4,98.0,1.3,NaN,135.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4976403,19989204,NaN,2188-03-31 16:00:00,49388416,3.2,NaN,NaN,14.0,23.0,54.0,8.3,98.0,2.1,111.0,135.0,5.0
4976404,19989280,NaN,2164-04-28 15:05:00,36803604,4.3,NaN,NaN,16.0,25.0,15.0,NaN,100.0,0.6,133.0,141.0,3.8
4976405,19989280,28563896.0,2164-05-13 06:10:00,76394110,NaN,NaN,NaN,14.0,26.0,9.0,8.1,105.0,0.4,94.0,145.0,3.0
4976406,19989302,NaN,2127-11-11 19:25:00,15500408,4.3,NaN,NaN,24.0,9.0,25.0,8.7,106.0,1.4,221.0,139.0,4.3


In [10]:
# 1. Định nghĩa các cột cần dùng
chem_cols = [
    'subject_id', 'hadm_id', 'charttime', 
    'creatinine', 'bun', 'bicarbonate', 'potassium', 'sodium', 
    'aniongap', 'albumin', 'glucose'
]

# 2. Đọc file
df_chem = pd.read_csv('measurement/chemistry.csv', usecols=lambda x: x in chem_cols)
df_chem['charttime'] = pd.to_datetime(df_chem['charttime'])

# 3. Merge với cohort
merged_chem = pd.merge(
    aki_df[['stay_id', 'hadm_id', 'intime']],
    df_chem,
    on=['hadm_id'],
    how='inner'
)

# 4. Lọc cửa sổ thời gian
valid_chem = merged_chem[
    (merged_chem['charttime'] >= merged_chem['intime'] - pd.Timedelta(hours=6)) & 
    (merged_chem['charttime'] <= merged_chem['intime'] + pd.Timedelta(hours=12))
].copy()

# 5. Aggregation
agg_dict = {
    'creatinine': ['min'],
    'bun':        ['max'],
    'bicarbonate':['min'],
    'potassium':  ['max'],
    'sodium':     ['median'],
    'aniongap':   ['max'],
    'albumin':    ['median'],
    'glucose':    ['max']
}

available_aggs = {k: v for k, v in agg_dict.items() if k in valid_chem.columns}
df_chem_features = valid_chem.groupby('stay_id').agg(available_aggs)
df_chem_features.columns = ['_'.join(col).strip() for col in df_chem_features.columns.values]
df_chem_features = df_chem_features.reset_index()

# Đổi tên các cột glucose thành glucose_chemistry
df_chem_features.columns = [col.replace('glucose_', 'glucose_chemistry_') if 'glucose_' in col else col for col in df_chem_features.columns]
df_chem_features.columns = [col.replace('potassium_', 'potassium_chemistry_') if 'potassium_' in col else col for col in df_chem_features.columns]
df_chem_features.columns = [col.replace('sodium_', 'sodium_chemistry_') if 'sodium_' in col else col for col in df_chem_features.columns]

# 6. Derived features - BUN/Creat ratio (xử lý chia cho 0)
if 'bun_mean' in df_chem_features.columns and 'creatinine_mean' in df_chem_features.columns:
    df_chem_features['bun_creat_ratio'] = df_chem_features['bun_mean'] / df_chem_features['creatinine_mean'].replace(0, np.nan)

# 7. Merge vào bảng chính
aki_df = pd.merge(aki_df, df_chem_features, on='stay_id', how='left')

aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,spo2_min,glucose_vitalsign_max,creatinine_min,bun_max,bicarbonate_min,potassium_chemistry_max,sodium_chemistry_median,aniongap_max,albumin_median,glucose_chemistry_max
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,89.0,95.0,0.9,19.0,25.0,4.5,139.0,13.0,NaN,95.0
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,94.0,331.0,1.4,35.0,14.0,4.1,129.0,23.0,NaN,331.0
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,97.0,93.0,0.7,10.0,21.0,4.0,140.0,13.0,3.6,93.0
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,93.0,214.0,0.8,13.0,27.0,3.4,139.0,7.0,NaN,NaN
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,97.0,111.0,0.8,11.0,22.0,3.7,138.0,17.0,NaN,111.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,93.0,166.0,1.1,23.0,26.0,4.2,142.0,13.0,NaN,166.0
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,92.0,177.0,1.5,38.0,23.0,4.5,130.5,19.0,NaN,177.0
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,96.0,203.0,1.0,12.0,23.0,4.4,139.0,10.0,NaN,NaN
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,95.0,174.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### URINE OUTPUT

In [11]:
pd.read_csv('measurement/urine_output.csv')

,stay_id,charttime,urineoutput
0,38661600,2148-08-17 06:00:00,45.0
1,38661600,2148-08-19 20:00:00,120.0
2,38661600,2148-08-19 23:00:00,35.0
3,38661600,2148-08-18 13:00:00,100.0
4,38661600,2148-08-18 15:00:00,120.0
...,...,...,...
4127629,32985767,2117-01-11 07:00:00,600.0
4127630,32985767,2117-01-11 09:00:00,180.0
4127631,32985767,2117-01-12 01:00:00,400.0
4127632,32985767,2117-01-06 13:00:00,75.0


In [12]:
# 1. Đọc file
df_uo = pd.read_csv('measurement/urine_output.csv')
df_uo['charttime'] = pd.to_datetime(df_uo['charttime'])

# 2. Merge với cohort
merged_uo = pd.merge(
    aki_df[['stay_id', 'intime']],
    df_uo,
    on='stay_id',
    how='inner'
)

# 3. Lọc dữ liệu trong 12h đầu
valid_uo = merged_uo[
    (merged_uo['charttime'] >= merged_uo['intime']) & 
    (merged_uo['charttime'] <= merged_uo['intime'] + pd.Timedelta(hours=12))
].copy()

valid_uo = valid_uo[valid_uo['urineoutput'] >= 0] # Bỏ giá trị âm

# 4. Aggregation: TÍNH TỔNG (SUM)
df_uo_features = valid_uo.groupby('stay_id')['urineoutput'].sum().reset_index()
df_uo_features.rename(columns={'urineoutput': 'urineoutput_12h_sum'}, inplace=True)

# 5. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_uo_features, on='stay_id', how='left')
aki_df['urineoutput_12h_sum'] = aki_df['urineoutput_12h_sum']

if 'weight' in aki_df.columns:
    safe_weight = aki_df['weight'].replace(0, np.nan)
    aki_df['uo_rate_ml_kg_hr'] = aki_df['urineoutput_12h_sum'] / safe_weight / 12

print("Đã xử lý xong Urine Output.")
print(f"Missing urine output: {aki_df['urineoutput_12h_sum'].isna().sum()} / {len(aki_df)}")

Đã xử lý xong Urine Output.
Missing urine output: 37 / 2316


In [13]:
aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,creatinine_min,bun_max,bicarbonate_min,potassium_chemistry_max,sodium_chemistry_median,aniongap_max,albumin_median,glucose_chemistry_max,urineoutput_12h_sum,uo_rate_ml_kg_hr
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,0.9,19.0,25.0,4.5,139.0,13.0,NaN,95.0,1390.0,2.185535
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,1.4,35.0,14.0,4.1,129.0,23.0,NaN,331.0,2955.0,3.841654
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,0.7,10.0,21.0,4.0,140.0,13.0,3.6,93.0,775.0,0.736412
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,0.8,13.0,27.0,3.4,139.0,7.0,NaN,NaN,460.0,0.421245
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,0.8,11.0,22.0,3.7,138.0,17.0,NaN,111.0,900.0,0.945776
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,1.1,23.0,26.0,4.2,142.0,13.0,NaN,166.0,450.0,0.506757
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,1.5,38.0,23.0,4.5,130.5,19.0,NaN,177.0,545.0,0.772392
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,1.0,12.0,23.0,4.4,139.0,10.0,NaN,NaN,595.0,0.435323
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,775.0,1.218553


### COMPLETE BLOOD COUNT

In [14]:
pd.read_csv('measurement/complete_blood_count.csv')

,subject_id,hadm_id,charttime,specimen_id,hematocrit,hemoglobin,mch,mchc,mcv,platelet,rbc,rdw,rdwsd,wbc
0,10216153,NaN,2156-12-31 10:22:00,35888136,32.7,11.4,30.3,34.9,87.0,241.0,3.75,13.5,NaN,2.5
1,10216153,NaN,2157-02-13 11:35:00,57378694,28.0,9.7,29.8,34.6,86.0,358.0,3.26,15.7,NaN,2.6
2,10216153,NaN,2159-11-29 10:00:00,39707320,27.7,9.9,30.9,35.7,87.0,221.0,3.20,17.0,NaN,1.3
3,10216153,NaN,2160-02-01 11:50:00,60988608,32.7,11.4,31.2,35.0,89.0,309.0,3.67,15.0,NaN,3.4
4,10427159,NaN,2151-01-01 08:00:00,13107578,31.9,10.6,30.3,33.3,91.0,237.0,3.51,13.6,NaN,5.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4377895,19989030,21613555.0,2119-03-14 05:35:00,18868100,25.3,8.8,30.6,34.9,88.0,161.0,2.88,12.9,NaN,7.1
4377896,19989126,22853928.0,2149-08-08 07:58:00,76505767,28.3,9.4,31.1,33.2,94.0,152.0,3.03,13.0,NaN,9.6
4377897,19989134,NaN,2111-04-11 12:35:00,18140174,24.1,8.4,35.9,34.8,103.0,75.0,2.34,18.8,NaN,2.9
4377898,19989302,NaN,2124-02-13 11:45:00,24648761,45.1,15.0,30.0,33.3,90.0,229.0,5.00,13.1,NaN,12.5


In [15]:
# 1. Định nghĩa cột cần lấy
cbc_cols = [
    'subject_id', 'hadm_id', 'charttime',
    'hemoglobin', 'hematocrit', 'wbc', 'platelet', 'rdw'
]

# 2. Đọc file
df_cbc = pd.read_csv('measurement/complete_blood_count.csv', usecols=lambda x: x in cbc_cols)
df_cbc['charttime'] = pd.to_datetime(df_cbc['charttime'])

# 3. Merge với cohort (theo hadm_id vì đây là Lab)
merged_cbc = pd.merge(
    aki_df[['stay_id', 'hadm_id', 'intime']],
    df_cbc,
    on=['hadm_id'],
    how='inner'
)

# 4. Lọc cửa sổ thời gian
valid_cbc = merged_cbc[
    (merged_cbc['charttime'] >= merged_cbc['intime'] - pd.Timedelta(hours=6)) &
    (merged_cbc['charttime'] <= merged_cbc['intime'] + pd.Timedelta(hours=12))
].copy()

# 5. Aggregation
agg_dict = {
    'hemoglobin': ['min'],
    'hematocrit': ['min'],
    'wbc':        ['max'],
    'platelet':   ['median'],
    'rdw':        ['median']
}

available_aggs = {k: v for k, v in agg_dict.items() if k in valid_cbc.columns}
df_cbc_features = valid_cbc.groupby('stay_id').agg(available_aggs)
df_cbc_features.columns = ['_'.join(col).strip() for col in df_cbc_features.columns.values]
df_cbc_features = df_cbc_features.reset_index()

# 6. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_cbc_features, on='stay_id', how='left')
aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,aniongap_max,albumin_median,glucose_chemistry_max,urineoutput_12h_sum,uo_rate_ml_kg_hr,hemoglobin_min,hematocrit_min,wbc_max,platelet_median,rdw_median
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,13.0,NaN,95.0,1390.0,2.185535,12.5,37.9,5.5,185.0,15.20
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,23.0,NaN,331.0,2955.0,3.841654,14.0,40.6,36.8,180.5,12.40
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,13.0,3.6,93.0,775.0,0.736412,10.3,32.0,8.7,199.0,13.80
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,7.0,NaN,NaN,460.0,0.421245,7.2,22.0,14.8,240.5,14.55
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,17.0,NaN,111.0,900.0,0.945776,12.5,37.8,12.5,205.0,13.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,13.0,NaN,166.0,450.0,0.506757,12.4,39.1,15.7,174.0,14.70
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,19.0,NaN,177.0,545.0,0.772392,NaN,33.8,NaN,294.0,NaN
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,10.0,NaN,NaN,595.0,0.435323,10.6,31.5,20.0,199.5,12.35
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,NaN,NaN,NaN,775.0,1.218553,NaN,NaN,NaN,NaN,NaN


### CARDIAC MARKER

In [16]:
pd.read_csv('measurement/cardiac_marker.csv')

,subject_id,hadm_id,charttime,specimen_id,troponin_t,ck_mb,ntprobnp
0,10000980,20897796.0,2193-08-15 06:40:00,45469872,0.05,6.0,NaN
1,10001884,21268656.0,2125-10-18 21:10:00,39235632,NaN,3.0,NaN
2,10002430,NaN,2129-01-07 13:35:00,62797735,0.02,3.0,NaN
3,10003502,20459702.0,2166-02-16 16:20:00,35488485,0.09,4.0,NaN
4,10004401,NaN,2140-07-22 06:35:00,2391107,0.18,5.0,12510.0
...,...,...,...,...,...,...,...
380126,19929847,21424210.0,2128-05-23 15:55:00,73124084,3.31,28.0,NaN
380127,19989783,NaN,2128-06-18 09:30:00,91582311,NaN,NaN,1654.0
380128,19990786,NaN,2148-03-26 15:15:00,14527996,NaN,4.0,NaN
380129,17412632,22709968.0,2148-11-26 10:15:00,22373008,0.08,16.0,NaN


In [17]:
# 1. Định nghĩa cột
cardiac_cols = [
    'subject_id', 'hadm_id', 'charttime',
    'troponin_t', 'ck_mb', 'ntprobnp'
]

# 2. Đọc file
df_cardiac = pd.read_csv('measurement/cardiac_marker.csv', usecols=lambda x: x in cardiac_cols)
df_cardiac['charttime'] = pd.to_datetime(df_cardiac['charttime'])

# 3. Merge với cohort (theo hadm_id)
merged_cardiac = pd.merge(
    aki_df[['stay_id', 'hadm_id', 'intime']],
    df_cardiac,
    on=['hadm_id'],
    how='inner'
)

# 4. Lọc cửa sổ thời gian
valid_cardiac = merged_cardiac[
    (merged_cardiac['charttime'] >= merged_cardiac['intime'] - pd.Timedelta(hours=6)) &
    (merged_cardiac['charttime'] <= merged_cardiac['intime'] + pd.Timedelta(hours=12))
].copy()

# 5. Aggregation (Men tim quan tâm nhất là đỉnh - MAX)
agg_dict = {
    'troponin_t': ['max'],
    'ck_mb':      ['max'],
    'ntprobnp':   ['max']
}

available_aggs = {k: v for k, v in agg_dict.items() if k in valid_cardiac.columns}
df_cardiac_features = valid_cardiac.groupby('stay_id').agg(available_aggs)
df_cardiac_features.columns = ['_'.join(col).strip() for col in df_cardiac_features.columns.values]
df_cardiac_features = df_cardiac_features.reset_index()

# 6. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_cardiac_features, on='stay_id', how='left')
aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,urineoutput_12h_sum,uo_rate_ml_kg_hr,hemoglobin_min,hematocrit_min,wbc_max,platelet_median,rdw_median,troponin_t_max,ck_mb_max,ntprobnp_max
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,1390.0,2.185535,12.5,37.9,5.5,185.0,15.20,3.99,107.0,NaN
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,2955.0,3.841654,14.0,40.6,36.8,180.5,12.40,1.75,96.0,NaN
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,775.0,0.736412,10.3,32.0,8.7,199.0,13.80,NaN,NaN,NaN
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,460.0,0.421245,7.2,22.0,14.8,240.5,14.55,NaN,NaN,NaN
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,900.0,0.945776,12.5,37.8,12.5,205.0,13.00,1.37,178.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,450.0,0.506757,12.4,39.1,15.7,174.0,14.70,1.94,151.0,NaN
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,545.0,0.772392,NaN,33.8,NaN,294.0,NaN,5.18,120.0,NaN
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,595.0,0.435323,10.6,31.5,20.0,199.5,12.35,NaN,NaN,NaN
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,775.0,1.218553,NaN,NaN,NaN,NaN,NaN,NaN,308.0,NaN


### coagulation

In [18]:
pd.read_csv('measurement/coagulation.csv')

,subject_id,hadm_id,charttime,specimen_id,d_dimer,fibrinogen,thrombin,inr,pt,ptt
0,10028480,NaN,2189-05-11 12:40:00,8888102,NaN,NaN,NaN,2.8,29.1,NaN
1,10028480,27170495.0,2192-01-30 07:30:00,68623427,NaN,NaN,NaN,3.0,32.4,39.8
2,10169328,26624180.0,2194-01-01 18:50:00,63047805,NaN,NaN,NaN,NaN,NaN,52.0
3,10169328,21339805.0,2194-10-01 01:30:00,26537797,NaN,NaN,NaN,NaN,NaN,46.9
4,10023486,28704531.0,2155-06-04 05:41:00,18679315,NaN,NaN,NaN,1.5,16.6,NaN
...,...,...,...,...,...,...,...,...,...,...
1991162,19971771,26230047.0,2117-05-18 00:33:00,74827207,NaN,NaN,NaN,2.0,21.5,75.2
1991163,19973083,21885760.0,2123-10-05 04:13:00,19788737,NaN,NaN,NaN,5.1,55.8,75.1
1991164,19971180,NaN,2132-09-29 09:00:00,138417,NaN,NaN,NaN,1.0,11.1,26.5
1991165,19972786,25062403.0,2200-05-31 03:54:00,21185294,NaN,NaN,NaN,1.2,13.6,30.8


In [19]:
# 1. Định nghĩa cột
coag_cols = [
    'subject_id', 'hadm_id', 'charttime',
    'inr', 'pt', 'ptt', 'fibrinogen', 'd_dimer'
]

# 2. Đọc file
df_coag = pd.read_csv('measurement/coagulation.csv', usecols=lambda x: x in coag_cols)
df_coag['charttime'] = pd.to_datetime(df_coag['charttime'])

# 3. Merge với cohort (theo hadm_id)
merged_coag = pd.merge(
    aki_df[['stay_id', 'hadm_id', 'intime']],
    df_coag,
    on=['hadm_id'],
    how='inner'
)

# 4. Lọc cửa sổ thời gian
valid_coag = merged_coag[
    (merged_coag['charttime'] >= merged_coag['intime'] - pd.Timedelta(hours=6)) &
    (merged_coag['charttime'] <= merged_coag['intime'] + pd.Timedelta(hours=12))
].copy()

# 4.5. Outlier cleaning
# valid_coag.loc[(valid_coag['inr'] < 0) | (valid_coag['inr'] > 20), 'inr'] = None
# valid_coag.loc[(valid_coag['pt'] < 0) | (valid_coag['pt'] > 150), 'pt'] = None
# valid_coag.loc[(valid_coag['ptt'] < 0) | (valid_coag['ptt'] > 300), 'ptt'] = None
# valid_coag.loc[(valid_coag['fibrinogen'] < 0) | (valid_coag['fibrinogen'] > 1000), 'fibrinogen'] = None
# valid_coag.loc[(valid_coag['d_dimer'] < 0) | (valid_coag['d_dimer'] > 50000), 'd_dimer'] = None

# 5. Aggregation
agg_dict = {
    'inr':        ['median'],
    'pt':         ['median'],
    'ptt':        ['median'],
    'fibrinogen': ['median'],
    'd_dimer':    ['median']
}

available_aggs = {k: v for k, v in agg_dict.items() if k in valid_coag.columns}
df_coag_features = valid_coag.groupby('stay_id').agg(available_aggs)
df_coag_features.columns = ['_'.join(col).strip() for col in df_coag_features.columns.values]
df_coag_features = df_coag_features.reset_index()

# 6. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_coag_features, on='stay_id', how='left')

aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,platelet_median,rdw_median,troponin_t_max,ck_mb_max,ntprobnp_max,inr_median,pt_median,ptt_median,fibrinogen_median,d_dimer_median
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,185.0,15.20,3.99,107.0,NaN,NaN,NaN,NaN,NaN,NaN
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,180.5,12.40,1.75,96.0,NaN,1.30,14.50,49.60,NaN,NaN
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,199.0,13.80,NaN,NaN,NaN,1.00,10.90,69.25,NaN,NaN
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,240.5,14.55,NaN,NaN,NaN,1.45,15.55,30.10,233.0,NaN
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,205.0,13.00,1.37,178.0,NaN,1.10,11.60,34.75,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,174.0,14.70,1.94,151.0,NaN,2.60,27.90,35.50,NaN,NaN
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,294.0,NaN,5.18,120.0,NaN,NaN,NaN,NaN,NaN,NaN
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,199.5,12.35,NaN,NaN,NaN,1.20,13.50,29.95,375.0,NaN
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,NaN,NaN,NaN,308.0,NaN,NaN,NaN,NaN,NaN,NaN


### ENZYME

In [20]:
pd.read_csv('measurement/enzyme.csv')

,subject_id,hadm_id,charttime,specimen_id,alt,alp,ast,amylase,bilirubin_total,bilirubin_direct,bilirubin_indirect,ck_cpk,ck_mb,ggt,ld_ldh
0,10095141,NaN,2158-02-06 10:00:00,35197389,18.0,72.0,23.0,NaN,0.4,NaN,NaN,NaN,NaN,NaN,NaN
1,10095655,22335037.0,2118-02-20 03:48:00,7441189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.0,NaN,NaN,NaN
2,10096413,NaN,2181-06-18 12:12:00,56201500,12.0,NaN,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10096581,NaN,2147-01-31 10:50:00,36765046,14.0,NaN,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10120959,22645182.0,2182-02-11 16:17:00,54808223,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.0,6.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2187055,19994218,NaN,2119-03-29 10:02:00,76444353,9.0,58.0,12.0,NaN,0.4,NaN,0.4,NaN,NaN,NaN,175.0
2187056,19994588,NaN,2192-01-06 11:50:00,46728528,11.0,65.0,18.0,78.0,0.8,NaN,NaN,NaN,NaN,NaN,151.0
2187057,19994772,29606061.0,2180-12-14 13:50:00,8453043,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,2.0,NaN,NaN
2187058,19996219,NaN,2170-06-08 09:00:00,51249016,54.0,NaN,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# 1. Định nghĩa cột
enzyme_cols = [
    'subject_id', 'hadm_id', 'charttime',
    'ck_cpk', 'ld_ldh',
    'alt', 'ast', 'bilirubin_total',
    'alp', 'ggt', 'amylase'
]

# 2. Đọc file
df_enzyme = pd.read_csv('measurement/enzyme.csv', usecols=lambda x: x in enzyme_cols)
df_enzyme['charttime'] = pd.to_datetime(df_enzyme['charttime'])

# 3. Merge với cohort
merged_enzyme = pd.merge(
    aki_df[['stay_id', 'hadm_id', 'intime']],
    df_enzyme,
    on=['hadm_id'],
    how='inner'
)

# 4. Lọc cửa sổ thời gian
valid_enzyme = merged_enzyme[
    (merged_enzyme['charttime'] >= merged_enzyme['intime'] - pd.Timedelta(hours=6)) &
    (merged_enzyme['charttime'] <= merged_enzyme['intime'] + pd.Timedelta(hours=12))
].copy()

# 5. Aggregation
agg_dict = {
    'ck_cpk': ['max'],
    'ld_ldh': ['max'],
    'alt':    ['max'],
    'ast':    ['max'],
    'bilirubin_total': ['max'],
    'alp':    ['median'],
    'ggt':    ['median'],
    'amylase':['median']
}

available_aggs = {k: v for k, v in agg_dict.items() if k in valid_enzyme.columns}
df_enzyme_features = valid_enzyme.groupby('stay_id').agg(available_aggs)
df_enzyme_features.columns = ['_'.join(col).strip() for col in df_enzyme_features.columns.values]
df_enzyme_features = df_enzyme_features.reset_index()

# 6. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_enzyme_features, on='stay_id', how='left')

aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,fibrinogen_median,d_dimer_median,ck_cpk_max,ld_ldh_max,alt_max,ast_max,bilirubin_total_max,alp_median,ggt_median,amylase_median
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,NaN,NaN,589.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,NaN,NaN,NaN,310.0,NaN,NaN,NaN,NaN,NaN,NaN
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,NaN,NaN,NaN,NaN,40.0,32.0,0.5,97.0,NaN,NaN
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,233.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,NaN,NaN,2432.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,NaN,NaN,1156.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,375.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### GCS

In [22]:
pd.read_csv('measurement/gcs.csv')

,subject_id,stay_id,charttime,gcs,gcs_motor,gcs_verbal,gcs_eyes,gcs_unable
0,12435705,37677177,2125-04-17 07:00:00,13.0,6.0,4.0,3.0,0
1,12435705,37677177,2125-04-17 08:00:00,13.0,6.0,4.0,3.0,0
2,12435705,37677177,2125-04-17 12:00:00,13.0,6.0,4.0,3.0,0
3,12435705,37677177,2125-04-17 17:00:00,15.0,1.0,0.0,1.0,1
4,12435705,37677177,2125-04-17 18:03:00,15.0,6.0,0.0,2.0,1
...,...,...,...,...,...,...,...,...
2217782,10792782,31253530,2192-06-26 20:00:00,15.0,1.0,0.0,1.0,1
2217783,10792782,31253530,2192-07-04 15:00:00,15.0,1.0,0.0,1.0,1
2217784,10792782,31253530,2192-07-13 08:00:00,15.0,6.0,0.0,4.0,1
2217785,12110838,31314097,2158-04-04 20:00:00,9.0,4.0,2.0,3.0,0


In [23]:
# 1. Định nghĩa cột
gcs_cols = [
    'subject_id', 'stay_id', 'charttime',
    'gcs', 'gcs_motor', 'gcs_unable'
]

# 2. Đọc file
df_gcs = pd.read_csv('measurement/gcs.csv', usecols=lambda x: x in gcs_cols)
df_gcs['charttime'] = pd.to_datetime(df_gcs['charttime'])

# 3. Merge với cohort
merged_gcs = pd.merge(
    aki_df[['stay_id', 'intime']],
    df_gcs,
    on=['stay_id'],
    how='inner'
)

# 4. Lọc cửa sổ thời gian
valid_gcs = merged_gcs[
    (merged_gcs['charttime'] >= merged_gcs['intime']) &
    (merged_gcs['charttime'] <= merged_gcs['intime'] + pd.Timedelta(hours=12))
].copy()

# 5. Aggregation
agg_dict = {
    'gcs':        ['min'],
    'gcs_motor':  ['min'],
    'gcs_unable': ['max']
}

available_aggs = {k: v for k, v in agg_dict.items() if k in valid_gcs.columns}
df_gcs_features = valid_gcs.groupby('stay_id').agg(available_aggs)
df_gcs_features.columns = ['_'.join(col).strip() for col in df_gcs_features.columns.values]
df_gcs_features = df_gcs_features.reset_index()

# 6. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_gcs_features, on='stay_id', how='left')

aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,ld_ldh_max,alt_max,ast_max,bilirubin_total_max,alp_median,ggt_median,amylase_median,gcs_min,gcs_motor_min,gcs_unable_max
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,6.0,0.0
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,310.0,NaN,NaN,NaN,NaN,NaN,NaN,15.0,6.0,0.0
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,NaN,40.0,32.0,0.5,97.0,NaN,NaN,15.0,6.0,0.0
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0,1.0,1.0
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,6.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,6.0,0.0
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,6.0,0.0
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,1.0,1.0
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0,6.0,0.0


### VENT

In [24]:
pd.read_csv('measurement/ventilator_setting.csv')

,subject_id,stay_id,charttime,respiratory_rate_set,respiratory_rate_total,respiratory_rate_spontaneous,minute_volume,tidal_volume_set,tidal_volume_observed,tidal_volume_spontaneous,plateau_pressure,peep,fio2,flow_rate,ventilator_mode,ventilator_mode_hamilton,ventilator_type
0,10011427,33630048,2136-01-25 16:00:00,NaN,14.0,14.0,5.8,NaN,408.0,408.0,NaN,5.0,35.0,36.7,NaN,SPONT,Hamilton
1,10012292,35495588,2171-05-25 22:09:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN,NaN,NaN,NaN
2,10032381,34622731,2115-08-05 00:00:00,24.0,32.0,0.0,13.2,360.0,359.0,NaN,NaN,8.0,50.0,NaN,CMV/ASSIST/AutoFlow,NaN,Drager
3,10058575,32245180,2157-01-02 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN,NaN,NaN,NaN
4,10131457,38790233,2143-03-21 08:00:00,18.0,18.0,NaN,7.1,450.0,415.0,NaN,16.0,5.0,35.0,NaN,CMV/ASSIST,NaN,Drager
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377509,19993726,31439824,2166-09-19 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN
1377510,19993726,31439824,2166-09-21 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN
1377511,19994505,35505750,2185-11-04 08:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,CPAP/PSV,NaN,NaN
1377512,19995091,32457489,2172-08-25 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN,NaN,NaN,NaN


In [25]:
# 1. Định nghĩa cột
vent_cols = [
    'stay_id', 'charttime',
    'peep', 'fio2', 'plateau_pressure', 'tidal_volume_observed'
]

# 2. Đọc file
df_vent = pd.read_csv('measurement/ventilator_setting.csv', usecols=lambda x: x in vent_cols)
df_vent['charttime'] = pd.to_datetime(df_vent['charttime'])

# 3. Merge với cohort
merged_vent = pd.merge(
    aki_df[['stay_id', 'intime']],
    df_vent,
    on=['stay_id'],
    how='inner'
)

# 4. Lọc cửa sổ thời gian
valid_vent = merged_vent[
    (merged_vent['charttime'] >= merged_vent['intime']) &
    (merged_vent['charttime'] <= merged_vent['intime'] + pd.Timedelta(hours=12))
].copy()

# 5. Aggregation
agg_dict = {
    'peep': ['max'],
    'fio2': ['max'],
    'plateau_pressure': ['max'],
    'tidal_volume_observed': ['max']
}

available_aggs = {k: v for k, v in agg_dict.items() if k in valid_vent.columns}
df_vent_features = valid_vent.groupby('stay_id').agg(available_aggs)
df_vent_features.columns = ['_'.join(col).strip() for col in df_vent_features.columns.values]
df_vent_features = df_vent_features.reset_index()

# 6. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_vent_features, on='stay_id', how='left')

aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,alp_median,ggt_median,amylase_median,gcs_min,gcs_motor_min,gcs_unable_max,peep_max,fio2_max,plateau_pressure_max,tidal_volume_observed_max
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,NaN,NaN,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,NaN,NaN,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,97.0,NaN,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,NaN,NaN,NaN,14.0,1.0,1.0,5.0,100.0,NaN,410.0
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,NaN,NaN,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,NaN,NaN,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,NaN,NaN,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,NaN,NaN,NaN,3.0,1.0,1.0,8.0,100.0,NaN,587.0
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,NaN,NaN,NaN,14.0,6.0,0.0,NaN,NaN,NaN,NaN


### NOREPINEPHRINE

In [26]:
pd.read_csv('medication/norepinephrine_equivalent_dose.csv')

,stay_id,starttime,endtime,norepinephrine_equivalent_dose
0,36549179,2120-05-08 22:03:00,2120-05-08 22:57:00,0.200
1,36549179,2120-05-08 20:30:00,2120-05-08 21:02:00,0.602
2,36549179,2120-05-08 21:45:00,2120-05-08 22:03:00,0.301
3,39304348,2165-05-01 17:07:00,2165-05-01 17:19:00,0.090
4,37251786,2132-10-22 17:37:00,2132-10-22 19:30:00,0.494
...,...,...,...,...
783608,38939010,2190-09-17 01:03:00,2190-09-17 06:05:00,0.030
783609,38939010,2190-09-17 07:08:00,2190-09-17 08:52:00,0.010
783610,38939010,2190-09-17 06:05:00,2190-09-17 07:08:00,0.020
783611,38907664,2187-09-26 14:07:00,2187-09-26 14:13:00,0.050


In [27]:
# 1. Đọc file
df_vaso = pd.read_csv('medication/norepinephrine_equivalent_dose.csv')
df_vaso['starttime'] = pd.to_datetime(df_vaso['starttime'])
df_vaso['endtime'] = pd.to_datetime(df_vaso['endtime'])

# 2. Merge với cohort
merged_vaso = pd.merge(
    aki_df[['stay_id', 'intime']],
    df_vaso,
    on='stay_id',
    how='inner'
)

# 3. Lọc các liều thuốc có giao thoa với cửa sổ 12h
# Logic: Thuốc bắt đầu TRƯỚC khi hết 12h VÀ kết thúc SAU khi bệnh nhân vào ICU
valid_vaso = merged_vaso[
    (merged_vaso['starttime'] <= merged_vaso['intime'] + pd.Timedelta(hours=12)) &
    (merged_vaso['endtime'] >= merged_vaso['intime'])
].copy()

# 4. Aggregation
agg_dict = {
    'norepinephrine_equivalent_dose': ['max']
}

available_aggs = {k: v for k, v in agg_dict.items() if k in valid_vaso.columns}
df_vaso_features = valid_vaso.groupby('stay_id').agg(available_aggs)
df_vaso_features.columns = ['_'.join(col).strip() for col in df_vaso_features.columns.values]
df_vaso_features = df_vaso_features.reset_index()

# Tạo biến flag
df_vaso_features['vaso_flag'] = 1

# 5. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_vaso_features, on='stay_id', how='left')

# 6. Xử lý missing data - Điền 0 cho flag
aki_df['vaso_flag'] = aki_df['vaso_flag'].fillna(0)

# Điền 0 cho liều lượng (Nếu không dùng thuốc nghĩa là liều = 0)
if 'norepinephrine_equivalent_dose_max' in aki_df.columns:
    aki_df['norepinephrine_equivalent_dose_max'] = aki_df['norepinephrine_equivalent_dose_max'].fillna(0)
    
if 'norepinephrine_equivalent_dose_mean' in aki_df.columns:
    aki_df['norepinephrine_equivalent_dose_mean'] = aki_df['norepinephrine_equivalent_dose_mean'].fillna(0)

print("Đã xử lý xong Vasopressor.")
print(f"Số BN dùng vận mạch: {aki_df['vaso_flag'].sum()} / {len(aki_df)}")

# Check thử xem có ai liều = 0 mà flag = 1 không (sanity check)
invalid_check = aki_df[(aki_df['vaso_flag'] == 1) & (aki_df['norepinephrine_equivalent_dose_max'] == 0)]
if not invalid_check.empty:
    print(f"Cảnh báo: Có {len(invalid_check)} ca có Flag=1 nhưng Liều Max=0 (Cần xem lại dữ liệu gốc)")

aki_df

Đã xử lý xong Vasopressor.
Số BN dùng vận mạch: 732.0 / 2316


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,amylase_median,gcs_min,gcs_motor_min,gcs_unable_max,peep_max,fio2_max,plateau_pressure_max,tidal_volume_observed_max,norepinephrine_equivalent_dose_max,vaso_flag
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN,0.04,1.0
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,NaN,14.0,1.0,1.0,5.0,100.0,NaN,410.0,0.05,1.0
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,NaN,15.0,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,NaN,3.0,1.0,1.0,8.0,100.0,NaN,587.0,0.03,1.0
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,NaN,14.0,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0


### NSAID

In [28]:
pd.read_csv('medication/nsaid.csv')

,subject_id,hadm_id,nsaid,starttime,stoptime
0,10424034,26179943,Aspirin EC,2159-12-24 18:00:00,2159-12-25 16:00:00
1,10424034,26179943,Aspirin EC,2159-12-26 08:00:00,2160-01-01 00:00:00
2,10424034,26179943,Aspirin,2159-12-26 08:00:00,2159-12-26 07:00:00
3,10424041,27875455,Aspirin,2129-11-12 10:00:00,2129-11-17 15:00:00
4,10424041,27875455,Aspirin,2129-11-17 10:00:00,2129-11-25 20:00:00
...,...,...,...,...,...
293248,19985513,26979819,Aspirin,2120-05-06 10:00:00,2120-05-07 09:00:00
293249,19985513,26979819,Aspirin EC,2120-05-04 10:00:00,2120-05-05 14:00:00
293250,19985545,20159218,Aspirin,2140-03-18 08:00:00,2140-03-18 17:00:00
293251,19985545,20159218,Aspirin,2140-03-18 18:00:00,2140-03-22 18:00:00


In [29]:
import pandas as pd
import numpy as np

df_nsaid = pd.read_csv('medication/nsaid.csv')
df_nsaid['starttime'] = pd.to_datetime(df_nsaid['starttime'])
df_nsaid['stoptime'] = pd.to_datetime(df_nsaid['stoptime'])

merged_nsaid = pd.merge(aki_df[['stay_id', 'hadm_id', 'intime']], df_nsaid, on=['hadm_id'], how='inner')

valid_nsaid = merged_nsaid[
    (merged_nsaid['starttime'] <= merged_nsaid['intime'] + pd.Timedelta(hours=12)) & 
    (merged_nsaid['stoptime'] >= merged_nsaid['intime'])
].copy()

def classify_nsaid_row(drug_name):
    s = str(drug_name).lower().strip()
    is_aspirin = 0
    if 'aspirin' in s:
        is_aspirin = 1
    return is_aspirin

valid_nsaid['use_aspirin'] = valid_nsaid['nsaid'].apply(classify_nsaid_row)

nsaid_flags = valid_nsaid.groupby('stay_id')['use_aspirin'].max().reset_index()

aki_df = pd.merge(aki_df, nsaid_flags, on='stay_id', how='left')
aki_df['use_aspirin'] = aki_df['use_aspirin'].fillna(0).astype(int)

print("Thống kê biến Aspirin:")
print(aki_df['use_aspirin'].value_counts())
print("\nCheck mẫu 5 dòng có dùng Aspirin:")
print(aki_df[aki_df['use_aspirin'] == 1][['stay_id', 'use_aspirin']].head())

Thống kê biến Aspirin:
use_aspirin
1    1613
0     703
Name: count, dtype: int64

Check mẫu 5 dòng có dùng Aspirin:
    stay_id  use_aspirin
0  33685454            1
1  36753294            1
3  32604416            1
4  32506122            1
5  34139812            1


### ACEI

In [30]:
pd.read_csv('medication/acei.csv')

,subject_id,hadm_id,acei,starttime,stoptime
0,10303081,21147888,Lisinopril,2134-09-10 10:00:00,2134-09-13 22:00:00
1,10303081,24805005,Lisinopril,2134-02-06 10:00:00,2134-02-08 18:00:00
2,10303081,28854743,Lisinopril,2134-09-16 10:00:00,2134-09-18 22:00:00
3,10303361,25112963,Lisinopril,2187-03-18 08:00:00,2187-03-18 04:00:00
4,10303398,25161837,Lisinopril,2154-02-08 08:00:00,2154-02-08 07:00:00
...,...,...,...,...,...
135148,19985349,24547553,Lisinopril,2127-09-19 08:00:00,2127-09-19 12:00:00
135149,19985349,24547553,Lisinopril,2127-09-19 08:00:00,2127-09-19 09:00:00
135150,19985513,26979819,Lisinopril,2120-05-09 10:00:00,2120-05-09 14:00:00
135151,19985513,26979819,Lisinopril,2120-05-09 10:00:00,2120-05-10 20:00:00


In [31]:
df_acei = pd.read_csv('medication/acei.csv')
df_acei['starttime'] = pd.to_datetime(df_acei['starttime'])
df_acei['stoptime'] = pd.to_datetime(df_acei['stoptime'])

merged_acei = pd.merge(
    aki_df[['stay_id', 'hadm_id', 'intime']],
    df_acei,
    on=['hadm_id'],
    how='inner'
)

valid_acei = merged_acei[
    (merged_acei['starttime'] <= merged_acei['intime'] + pd.Timedelta(hours=12)) &
    (merged_acei['stoptime'] >= merged_acei['intime'])
].copy()

acei_flag = valid_acei.groupby('stay_id').size().reset_index(name='has_acei')
acei_flag['has_acei'] = 1
acei_flag = acei_flag[['stay_id', 'has_acei']]

aki_df = pd.merge(aki_df, acei_flag[['stay_id', 'has_acei']], on='stay_id', how='left')
aki_df['has_acei'] = aki_df['has_acei'].fillna(0).astype(int)

print(f"Số ca dùng ACEI: {aki_df['has_acei'].sum()}")
aki_df

Số ca dùng ACEI: 353


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,gcs_motor_min,gcs_unable_max,peep_max,fio2_max,plateau_pressure_max,tidal_volume_observed_max,norepinephrine_equivalent_dose_max,vaso_flag,use_aspirin,has_acei
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0,1,1
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,6.0,0.0,NaN,NaN,NaN,NaN,0.04,1.0,1,0
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0,0,0
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,1.0,1.0,5.0,100.0,NaN,410.0,0.05,1.0,1,0
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0,1,0
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0,1,0
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,1.0,1.0,8.0,100.0,NaN,587.0,0.03,1.0,0,0
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,6.0,0.0,NaN,NaN,NaN,NaN,0.00,0.0,1,0


### ANTIBIOTIC

In [32]:
pd.read_csv('medication/antibiotic.csv')    

,subject_id,hadm_id,stay_id,antibiotic,route,starttime,stoptime
0,10595935,22439475,NaN,CefTRIAXone,IV,2166-11-23 18:00:00,2166-12-01 23:00:00
1,10595935,22439475,NaN,CefTRIAXone,IV,2166-11-20 00:00:00,2166-11-21 17:00:00
2,10595935,22439475,NaN,Vancomycin,IV,2166-11-20 00:00:00,2166-11-23 09:00:00
3,10595935,24192105,NaN,CefTRIAXone,IV,2167-02-20 07:00:00,2167-02-22 18:00:00
4,10595935,24192105,NaN,Gentamicin,IV,2167-02-16 19:00:00,2167-02-17 12:00:00
...,...,...,...,...,...,...,...
949896,19985409,27293537,32597806.0,Vancomycin,IV,2154-09-22 08:00:00,2154-09-25 21:00:00
949897,19985409,27293537,32597806.0,CefTRIAXone,IV,2154-09-23 15:00:00,2154-09-28 19:00:00
949898,19985409,27293537,32597806.0,MetroNIDAZOLE,PO/NG,2154-09-23 20:00:00,2154-09-28 19:00:00
949899,19985409,27293537,32597806.0,CefePIME,IV,2154-09-22 20:00:00,2154-09-23 14:00:00


In [33]:
import pandas as pd
import numpy as np

df_abx = pd.read_csv('medication/antibiotic.csv')
df_abx['starttime'] = pd.to_datetime(df_abx['starttime'])
df_abx['stoptime'] = pd.to_datetime(df_abx['stoptime'])

if 'stay_id' in df_abx.columns:
    merge_cols = ['stay_id']
    cohort_cols = ['stay_id', 'intime']
else:
    merge_cols = ['hadm_id']
    cohort_cols = ['stay_id', 'hadm_id', 'intime']

merged_abx = pd.merge(
    aki_df[cohort_cols],
    df_abx,
    on=merge_cols,
    how='inner'
)

valid_abx = merged_abx[
    (merged_abx['starttime'] <= merged_abx['intime'] + pd.Timedelta(hours=12)) &
    (merged_abx['stoptime'] >= merged_abx['intime'])
].copy()

valid_abx['antibiotic_lower'] = valid_abx['antibiotic'].astype(str).str.lower().str.strip()
valid_abx['route_upper'] = valid_abx['route'].astype(str).str.upper().str.strip()

excluded_routes = ['NU', 'IN', 'PR', 'TP']
excluded_keywords = ['ointment', 'enema']

mask_systemic = (
    ~valid_abx['route_upper'].isin(excluded_routes) & 
    ~valid_abx['antibiotic_lower'].str.contains('|'.join(excluded_keywords))
)

systemic_abx = valid_abx[mask_systemic].copy()

systemic_abx['is_vanco_iv'] = (
    systemic_abx['antibiotic_lower'].str.contains('vancomycin') & 
    (systemic_abx['route_upper'] == 'IV')
).astype(int)

abx_features = systemic_abx.groupby('stay_id').agg(
    use_vancomycin_iv=('is_vanco_iv', 'max'),
    cnt_systemic_antibiotics=('antibiotic', 'nunique')
).reset_index()

aki_df = pd.merge(aki_df, abx_features, on='stay_id', how='left')

fill_cols = ['use_vancomycin_iv', 'cnt_systemic_antibiotics']
aki_df[fill_cols] = aki_df[fill_cols].fillna(0).astype(int)

print("--- Thống kê Kháng sinh ---")
print(aki_df[fill_cols].sum())
print("\n--- Mẫu dữ liệu sau khi xử lý ---")
print(aki_df[['stay_id'] + fill_cols].head())

--- Thống kê Kháng sinh ---
use_vancomycin_iv            707
cnt_systemic_antibiotics    1744
dtype: int64

--- Mẫu dữ liệu sau khi xử lý ---
    stay_id  use_vancomycin_iv  cnt_systemic_antibiotics
0  33685454                  0                         0
1  36753294                  0                         3
2  31573075                  0                         0
3  32604416                  1                         1
4  32506122                  0                         0


### INVASIVE LINE

In [34]:
pd.read_csv('treatment/invasive_line.csv')

,stay_id,line_type,line_site,starttime,endtime
0,30000153,Arterial,NaN,2174-09-29 13:15:00,2174-09-30 19:00:00
1,30000484,Multi Lumen,NaN,2136-01-15 18:17:00,2136-01-17 04:53:00
2,30001148,Arterial,Right Radial,2156-08-30 14:30:00,2156-08-31 08:05:00
3,30001148,Multi Lumen,NaN,2156-08-30 14:30:00,2156-08-31 14:25:00
4,30001446,Arterial,Right Radial,2186-04-12 06:00:00,2186-04-13 17:16:00
...,...,...,...,...,...
108160,39999301,Arterial,NaN,2111-08-18 17:12:00,2111-08-19 19:18:00
108161,39999384,Arterial,NaN,2158-05-24 21:00:00,2158-05-25 10:19:00
108162,39999552,Continuous Cardiac Output PA,Right IJ,2186-07-17 17:17:00,2186-07-18 10:15:00
108163,39999552,Cordis/Introducer,Right IJ,2186-07-17 17:17:00,2186-07-18 17:00:00


In [35]:
import pandas as pd
import numpy as np

df_inv = pd.read_csv('treatment/invasive_line.csv')
df_inv['starttime'] = pd.to_datetime(df_inv['starttime'])
df_inv['endtime'] = pd.to_datetime(df_inv['endtime'])

merged_inv = pd.merge(
    aki_df[['stay_id', 'intime']],
    df_inv,
    on=['stay_id'],
    how='inner'
)

valid_inv = merged_inv[
    (merged_inv['starttime'] <= merged_inv['intime'] + pd.Timedelta(hours=12)) &
    (merged_inv['endtime'] >= merged_inv['intime'])
].copy()

valid_inv['type_lower'] = valid_inv['line_type'].astype(str).str.lower().str.strip()
valid_inv['site_lower'] = valid_inv['line_site'].fillna('').astype(str).str.lower().str.strip()

valid_inv['is_mcs'] = valid_inv['type_lower'].str.contains('iabp|impella|tandem').astype(int)
valid_inv['is_pa'] = (
    (valid_inv['type_lower'] == 'pa') | 
    (valid_inv['type_lower'].str.contains('cardiac output pa'))
).astype(int)
valid_inv['is_femoral'] = valid_inv['site_lower'].str.contains('femoral').astype(int)

inv_features = valid_inv.groupby('stay_id').agg(
    has_mcs_device=('is_mcs', 'max'),
    has_pa_catheter=('is_pa', 'max'),
    is_femoral_access=('is_femoral', 'max'),
    cnt_invasive_lines=('line_type', 'nunique')
).reset_index()

aki_df = pd.merge(aki_df, inv_features, on='stay_id', how='left')

fill_cols = ['has_mcs_device', 'has_pa_catheter', 'is_femoral_access', 'cnt_invasive_lines']
aki_df[fill_cols] = aki_df[fill_cols].fillna(0).astype(int)

print(aki_df[fill_cols].sum())
print(aki_df['cnt_invasive_lines'].value_counts().sort_index())
print(aki_df[aki_df['has_mcs_device'] == 1][['stay_id', 'has_mcs_device', 'is_femoral_access', 'cnt_invasive_lines']].head())

has_mcs_device         250
has_pa_catheter        388
is_femoral_access      403
cnt_invasive_lines    2407
dtype: int64
cnt_invasive_lines
0    1175
1     233
2     617
3     238
4      41
5      10
6       2
Name: count, dtype: int64
     stay_id  has_mcs_device  is_femoral_access  cnt_invasive_lines
2   31573075               1                  1                   2
13  35736966               1                  0                   2
17  35746964               1                  1                   1
23  39342860               1                  1                   2
36  33394092               1                  1                   4


### VENTILATION

In [36]:
pd.read_csv('treatment/ventilation.csv')

,stay_id,starttime,endtime,ventilation_status
0,31889899,2174-01-28 23:38:00,2174-01-30 10:00:00,SupplementalOxygen
1,31945300,2158-03-02 19:00:00,2158-03-03 09:00:00,HFNC
2,31950308,2127-10-30 23:01:00,2127-11-01 15:47:00,InvasiveVent
3,31984434,2147-12-22 09:01:00,2147-12-22 18:00:00,SupplementalOxygen
4,31985218,2146-08-14 06:30:00,2146-08-15 04:00:00,InvasiveVent
...,...,...,...,...
144807,38517898,2154-11-11 15:00:00,2154-11-11 16:00:00,SupplementalOxygen
144808,38518526,2138-10-02 20:16:00,2138-10-03 09:00:00,SupplementalOxygen
144809,38529576,2139-04-29 02:00:00,2139-04-29 15:00:00,SupplementalOxygen
144810,38546783,2166-06-23 06:00:00,2166-06-23 09:00:00,NonInvasiveVent


In [37]:
# 1. Đọc file
df_vent = pd.read_csv('treatment/ventilation.csv')
df_vent['starttime'] = pd.to_datetime(df_vent['starttime'])
df_vent['endtime'] = pd.to_datetime(df_vent['endtime'])

# 2. Merge với cohort để lấy intime
merged_vent = pd.merge(
    aki_df[['stay_id', 'intime']],
    df_vent,
    on=['stay_id'],
    how='inner'
)

# 3. Lọc sự kiện thở máy trong 12h đầu
# Logic: Bắt đầu TRƯỚC khi hết 12h VÀ kết thúc SAU khi vào ICU
valid_vent = merged_vent[
    (merged_vent['starttime'] <= merged_vent['intime'] + pd.Timedelta(hours=12)) &
    (merged_vent['endtime'] >= merged_vent['intime'])
].copy()

# 4. Tạo biến Flag dựa trên ventilation_status
# Tạo cột Flag cho Invasive (Xâm lấn)
valid_vent['is_invasive'] = valid_vent['ventilation_status'].apply(
    lambda x: 1 if x in ['InvasiveVent', 'Tracheostomy'] else 0
)

# Tạo cột Flag cho Non-Invasive (Không xâm lấn)
valid_vent['is_non_invasive'] = valid_vent['ventilation_status'].apply(
    lambda x: 1 if x == 'NonInvasiveVent' else 0
)

# 5. Aggregation (Gom nhóm theo stay_id và lấy MAX)
# Lấy Max để bắt được tình trạng nặng nhất
agg_dict = {
    'is_invasive': 'max',
    'is_non_invasive': 'max'
}

df_vent_features = valid_vent.groupby('stay_id').agg(agg_dict).reset_index()

# Đổi tên cột
df_vent_features.columns = ['stay_id', 'invasive_vent_flag', 'non_invasive_vent_flag']

# 6. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_vent_features, on='stay_id', how='left')

# 7. Fillna bằng 0 (Không có record = tự thở hoặc oxy thường)
aki_df[['invasive_vent_flag', 'non_invasive_vent_flag']] = \
    aki_df[['invasive_vent_flag', 'non_invasive_vent_flag']].fillna(0)

aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,use_aspirin,has_acei,use_vancomycin_iv,cnt_systemic_antibiotics,has_mcs_device,has_pa_catheter,is_femoral_access,cnt_invasive_lines,invasive_vent_flag,non_invasive_vent_flag
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,1,1,0,0,0,0,1,2,0.0,0.0
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,1,0,0,3,0,0,0,0,0.0,0.0
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,0,0,0,0,1,0,1,2,0.0,0.0
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,1,0,1,1,0,1,0,3,0.0,0.0
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,1,0,0,0,0,0,0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,1,0,0,0,0,0,0,0,0.0,0.0
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,1,0,0,0,0,0,0,0,0.0,0.0
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,0,0,1,2,0,0,0,2,1.0,0.0
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,1,0,0,0,0,0,0,0,0.0,0.0


### SUSPICION OF INFECTION

In [38]:
pd.read_csv('sepsis/suspicion_of_infection.csv')

,subject_id,stay_id,hadm_id,ab_id,antibiotic,antibiotic_time,suspected_infection,suspected_infection_time,culture_time,specimen,positive_culture
0,17084991,NaN,25729037,1,Trimethoprim,2158-02-03 23:00:00,0,NaN,NaN,NaN,NaN
1,17087467,NaN,21559018,1,Ciprofloxacin HCl,2146-01-11 00:00:00,1,2146-01-10 22:00:00,2146-01-10 22:00:00,URINE,1.0
2,17087467,35498581.0,21559018,2,Cefpodoxime Proxetil,2146-01-14 10:00:00,0,NaN,NaN,NaN,NaN
3,17087467,35498581.0,21559018,3,Cefpodoxime Proxetil,2146-01-14 10:00:00,0,NaN,NaN,NaN,NaN
4,17087467,NaN,21559018,4,Cefpodoxime Proxetil,2146-01-15 10:00:00,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
949896,18012733,NaN,23003261,18,Amoxicillin-Clavulanic Acid,2171-08-09 13:00:00,1,2171-08-08 13:45:00,2171-08-08 13:45:00,BILE,0.0
949897,18012733,NaN,23003261,19,Piperacillin-Tazobactam,2171-08-09 18:00:00,1,2171-08-08 13:45:00,2171-08-08 13:45:00,BILE,0.0
949898,18012733,NaN,23003261,20,Ciprofloxacin HCl,2171-08-10 15:00:00,1,2171-08-08 13:45:00,2171-08-08 13:45:00,BILE,0.0
949899,18012733,NaN,23003261,21,MetroNIDAZOLE,2171-08-10 15:00:00,1,2171-08-08 13:45:00,2171-08-08 13:45:00,BILE,0.0


In [39]:
# 1. Đọc file
df_susp = pd.read_csv('sepsis/suspicion_of_infection.csv')
df_susp['suspected_infection_time'] = pd.to_datetime(df_susp['suspected_infection_time'])

# 2. Merge với cohort để lấy intime
merged_susp = pd.merge(
    aki_df[['stay_id', 'intime']],
    df_susp,
    on='stay_id',
    how='inner'
)

# 3. Lọc sự kiện nghi ngờ nhiễm trùng
# Logic: Thời điểm nghi ngờ xảy ra TRƯỚC khi hết 12h đầu
# (Lấy cả những ca nghi ngờ từ ED trước khi vào ICU)
valid_susp = merged_susp[
    (merged_susp['suspected_infection_time'] <= merged_susp['intime'] + pd.Timedelta(hours=12)) &
    (merged_susp['suspected_infection_time'] >= merged_susp['intime'] - pd.Timedelta(hours=24))  # NEW: Lower bound
].copy()

# ---------------------------------------------------------
# FEATURE 1: Biến Flag (Có nghi ngờ nhiễm trùng không?)
# ---------------------------------------------------------
df_susp_features = valid_susp.groupby('stay_id').size().reset_index(name='count')
df_susp_features['infection_suspected_flag'] = 1

# ---------------------------------------------------------
# FEATURE 2: Nguồn nhiễm trùng (Specimen) - Nâng cao
# ---------------------------------------------------------
# Kiểm tra xem có lấy máu (nghiêm trọng nhất) hay không
blood_mask = valid_susp['specimen'].str.contains('BLOOD', case=False, na=False)
df_blood = valid_susp[blood_mask].groupby('stay_id').size().reset_index(name='blood_culture_count')
df_blood['blood_culture_flag'] = 1

# 4. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_susp_features[['stay_id', 'infection_suspected_flag']], on='stay_id', how='left')
aki_df = pd.merge(aki_df, df_blood[['stay_id', 'blood_culture_flag']], on='stay_id', how='left')

# 5. Fillna bằng 0
aki_df['infection_suspected_flag'] = aki_df['infection_suspected_flag'].fillna(0)
aki_df['blood_culture_flag'] = aki_df['blood_culture_flag'].fillna(0)

aki_df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,use_vancomycin_iv,cnt_systemic_antibiotics,has_mcs_device,has_pa_catheter,is_femoral_access,cnt_invasive_lines,invasive_vent_flag,non_invasive_vent_flag,infection_suspected_flag,blood_culture_flag
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,0,0,0,0,1,2,0.0,0.0,1.0,0.0
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,0,3,0,0,0,0,0.0,0.0,1.0,0.0
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,0,0,1,0,1,2,0.0,0.0,0.0,0.0
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,1,1,0,1,0,3,0.0,0.0,1.0,0.0
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,1,2,0,0,0,2,1.0,0.0,0.0,0.0
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,0,0,0,0,0,0,0.0,0.0,1.0,0.0


### BLOOD DIFF

In [40]:
pd.read_csv('measurement/blood_differential.csv')

,subject_id,hadm_id,charttime,specimen_id,wbc,basophils_abs,eosinophils_abs,lymphocytes_abs,monocytes_abs,neutrophils_abs,basophils,eosinophils,lymphocytes,monocytes,neutrophils,atypical_lymphocytes,bands,immature_granulocytes,metamyelocytes,nrbc
0,10071129,22706742.0,2170-02-23 06:34:00,54006267,10.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10071129,NaN,2172-01-16 10:34:00,81052620,7.5,0.00,0.45,0.825,1.05,5.10,0.0,6.0,11.0,14.0,68.0,0.0,1.0,NaN,0.0,NaN
2,10071379,23735613.0,2127-09-09 06:55:00,50340820,10.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10071605,21911997.0,2156-08-17 20:25:00,67398476,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10071795,24331732.0,2173-04-21 22:30:00,65211864,9.7,0.04,0.18,2.280,0.60,6.52,0.4,1.9,23.6,6.2,67.6,NaN,NaN,0.3,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4154221,19995012,NaN,2166-04-10 09:30:00,67219317,12.2,0.09,0.20,6.040,0.92,4.92,0.7,1.6,49.5,7.5,40.4,NaN,NaN,0.3,NaN,NaN
4154222,19995832,23014132.0,2185-05-05 05:17:00,60051853,5.2,0.01,0.00,1.880,0.20,3.06,0.2,0.0,36.4,3.9,59.3,NaN,NaN,0.2,NaN,NaN
4154223,19995997,NaN,2124-09-20 14:23:00,65795686,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4154224,19996673,29017569.0,2149-06-09 14:50:00,31343354,10.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
# 1. Định nghĩa cột
diff_cols = [
    'subject_id', 'hadm_id', 'charttime',
    'neutrophils_abs', 'lymphocytes_abs', 
    'neutrophils', 'lymphocytes',         
    'bands'
]

# 2. Đọc file
df_diff = pd.read_csv('measurement/blood_differential.csv', usecols=lambda x: x in diff_cols)
df_diff['charttime'] = pd.to_datetime(df_diff['charttime'])

# --- BỔ SUNG: Data Cleaning (Rất quan trọng cho tỷ số) ---
# Loại bỏ giá trị âm hoặc % > 100
# for col in ['neutrophils', 'lymphocytes', 'bands']:
#     if col in df_diff.columns:
#         df_diff.loc[(df_diff[col] < 0) | (df_diff[col] > 100), col] = None

# # FIX: Add upper bounds for absolute counts
# for col in ['neutrophils_abs', 'lymphocytes_abs']:
#     if col in df_diff.columns:
#         df_diff.loc[(df_diff[col] < 0) | (df_diff[col] > 100), col] = None  # Cap at 100 K/µL

# 3. Tính chỉ số NLR (Feature Engineering)
# FIX: Safe division for absolute counts
df_diff['nlr'] = df_diff['neutrophils_abs'] / df_diff['lymphocytes_abs'].replace(0, np.nan)

# Fill bằng % nếu abs bị thiếu
mask_nan = df_diff['nlr'].isna()
mask_valid_pct = df_diff['neutrophils'].notna() & df_diff['lymphocytes'].notna()

# FIX: Safe division for percentages
safe_lymph_pct = df_diff.loc[mask_nan & mask_valid_pct, 'lymphocytes'].replace(0, np.nan)
df_diff.loc[mask_nan & mask_valid_pct, 'nlr'] = \
    df_diff.loc[mask_nan & mask_valid_pct, 'neutrophils'] / safe_lymph_pct

# Xử lý vô cực (chia cho 0) và giá trị quá lớn
df_diff['nlr'] = df_diff['nlr'].replace([np.inf, -np.inf], 100)
df_diff.loc[df_diff['nlr'] > 100, 'nlr'] = 100

# 4. Merge với cohort
merged_diff = pd.merge(
    aki_df[['stay_id', 'hadm_id', 'intime']],
    df_diff,
    on=['hadm_id'],
    how='inner'
)

# 5. Lọc cửa sổ thời gian (-6h đến +12h)
valid_diff = merged_diff[
    (merged_diff['charttime'] >= merged_diff['intime'] - pd.Timedelta(hours=6)) &
    (merged_diff['charttime'] <= merged_diff['intime'] + pd.Timedelta(hours=12))
].copy()

# 6. Aggregation
agg_dict = {
    'nlr':   ['max'],  
    'bands': ['max']
}

available_aggs = {k: v for k, v in agg_dict.items() if k in valid_diff.columns}
df_diff_features = valid_diff.groupby('stay_id').agg(available_aggs)

# Làm phẳng tên cột
df_diff_features.columns = ['_'.join(col).strip() for col in df_diff_features.columns.values]
df_diff_features = df_diff_features.reset_index()

# 7. Merge vào dataset chính
cols_to_drop = df_diff_features.columns.difference(['stay_id'])
if any(col in aki_df.columns for col in cols_to_drop):
    aki_df = aki_df.drop(columns=[c for c in cols_to_drop if c in aki_df.columns])

aki_df = pd.merge(aki_df, df_diff_features, on='stay_id', how='left')

# Check kết quả
print(f"Mean NLR Max: {aki_df['nlr_max'].mean():.2f}")
print(f"Số lượng có dữ liệu NLR: {aki_df['nlr_max'].notna().sum()} / {len(aki_df)}")

aki_df

Mean NLR Max: 7.01
Số lượng có dữ liệu NLR: 1136 / 2316


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,has_mcs_device,has_pa_catheter,is_femoral_access,cnt_invasive_lines,invasive_vent_flag,non_invasive_vent_flag,infection_suspected_flag,blood_culture_flag,nlr_max,bands_max
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,0,0,1,2,0.0,0.0,1.0,0.0,NaN,NaN
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,0,0,0,0,0.0,0.0,1.0,0.0,30.445455,10.0
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,1,0,1,2,0.0,0.0,0.0,0.0,NaN,NaN
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,0,1,0,3,0.0,0.0,1.0,0.0,9.121294,NaN
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,0,0,0,0,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,0,0,0,0,0.0,0.0,0.0,0.0,NaN,NaN
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,0,0,0,0,0.0,0.0,0.0,0.0,NaN,NaN
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,0,0,0,2,1.0,0.0,0.0,0.0,7.612745,NaN
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,0,0,0,0,0.0,0.0,1.0,0.0,NaN,NaN


### RHYTHM

In [42]:
pd.read_csv('measurement/rhythm.csv')

,subject_id,charttime,heart_rhythm,ectopy_type,ectopy_frequency,ectopy_type_secondary,ectopy_frequency_secondary
0,10022537,2185-02-05 23:00:00,A Paced,NaN,NaN,NaN,NaN
1,10022537,2185-02-06 00:00:00,A Paced,NaN,NaN,NaN,NaN
2,10022537,2185-02-07 07:00:00,AF (Atrial Fibrillation),NaN,NaN,NaN,NaN
3,10022537,2185-02-08 09:00:00,NaN,NaN,NaN,NaN,NaN
4,10022537,2185-02-18 11:00:00,ST (Sinus Tachycardia),NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
7887349,19991743,2130-10-06 11:00:00,SB (Sinus Bradycardia),PAC's,Rare,PVC's,Rare
7887350,19991743,2130-10-06 17:00:00,SB (Sinus Bradycardia),PAC's,Rare,PVC's,Rare
7887351,19992507,2178-11-05 01:40:00,ST (Sinus Tachycardia),NaN,NaN,NaN,NaN
7887352,19992875,2160-04-15 14:00:00,ST (Sinus Tachycardia),NaN,NaN,NaN,NaN


In [43]:
# 1. Đọc file
df_rhythm = pd.read_csv('measurement/rhythm.csv')
df_rhythm['charttime'] = pd.to_datetime(df_rhythm['charttime'])

# 2. Merge với cohort để lấy intime
merged_rhythm = pd.merge(
    aki_df[['subject_id', 'stay_id', 'intime']],
    df_rhythm,
    on=['subject_id'],
    how='inner'
)

# 3. Lọc cửa sổ thời gian (12h đầu)
valid_rhythm = merged_rhythm[
    (merged_rhythm['charttime'] >= merged_rhythm['intime'] - pd.Timedelta(hours=2)) &
    (merged_rhythm['charttime'] <= merged_rhythm['intime'] + pd.Timedelta(hours=12))
].copy()

# 4. Xử lý Text (Feature Engineering)
# Chuẩn hóa về chữ thường
valid_rhythm['rhythm_lower'] = valid_rhythm['heart_rhythm'].str.lower().fillna('')

# Tạo biến Flag - Pattern đã FIX để khớp với dữ liệu thực
valid_rhythm['is_afib'] = valid_rhythm['rhythm_lower'].str.contains('af |atrial fib|a flut|atrial flutter', na=False).astype(int)
valid_rhythm['is_vt_vf'] = valid_rhythm['rhythm_lower'].str.contains('vt |vf |ventricular tach|ventricular fib', na=False).astype(int)
valid_rhythm['is_paced'] = valid_rhythm['rhythm_lower'].str.contains('paced', na=False).astype(int)
high_block_pattern = r'3rd av|complete heart block|2nd av m2|mobitz 2'
valid_rhythm['is_high_grade_block'] = valid_rhythm['rhythm_lower'].str.contains(high_block_pattern, regex=True).astype(int)

# 5. Aggregation (Gom nhóm theo stay_id và lấy MAX)
# Chỉ cần xuất hiện 1 lần trong 12h là tính = 1
agg_dict = {
    'is_afib': 'max',
    'is_vt_vf': 'max',
    'is_paced': 'max',
    'is_high_grade_block': 'max'
}

df_rhythm_features = valid_rhythm.groupby('stay_id').agg(agg_dict).reset_index()

# 6. Merge vào dataset chính
aki_df = pd.merge(aki_df, df_rhythm_features, on='stay_id', how='left')

# 7. Fillna bằng 0
aki_df[['is_afib', 'is_vt_vf', 'is_paced', 'is_high_grade_block']] = \
    aki_df[['is_afib', 'is_vt_vf', 'is_paced', 'is_high_grade_block']].fillna(0)

# Kiểm tra kết quả
print(f"AFib/AFlutter: {aki_df['is_afib'].sum()}")
print(f"VT/VF: {aki_df['is_vt_vf'].sum()}")
print(f"Paced rhythm: {aki_df['is_paced'].sum()}")

aki_df

AFib/AFlutter: 150.0
VT/VF: 25.0
Paced rhythm: 541.0


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,admission_age,...,invasive_vent_flag,non_invasive_vent_flag,infection_suspected_flag,blood_culture_flag,nlr_max,bands_max,is_afib,is_vt_vf,is_paced,is_high_grade_block
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,81,...,0.0,0.0,1.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,0.0,0.0,1.0,0.0,30.445455,10.0,0.0,0.0,0.0,0.0
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,...,0.0,0.0,1.0,0.0,9.121294,NaN,0.0,0.0,1.0,0.0
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,78,...,0.0,0.0,0.0,0.0,NaN,NaN,1.0,0.0,1.0,0.0
2312,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,1.0
2313,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,...,1.0,0.0,0.0,0.0,7.612745,NaN,0.0,0.0,0.0,0.0
2314,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,84,...,0.0,0.0,1.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0


### SAP XEP LAI COT

In [44]:
# Lấy danh sách tất cả các cột thực tế hiện có trong DataFrame
cols = aki_df.columns.tolist()

# 1. Thông tin định danh & Label
id_cols = ['subject_id', 'hadm_id', 'stay_id', 'aki_label']

# 2. Thông tin nhân khẩu học & hành chính
demo_cols = [
    'gender', 'admission_age', 'race', 'weight', 'height',
    'intime', 'outtime', 
    'first_careunit', 'last_careunit',
    'hospstay_seq', 'first_hosp_stay',
    'prediction_start_time', 'prediction_end_time'
]

# 3. Vital Signs (Sinh hiệu)
vital_cols = [
    'heart_rate_max', 
    'sys_bp_min', 'mean_bp_min', 'dias_bp_min', 
    'resp_rate_max', 
    'temperature_max', 
    'spo2_min', 
    'glucose_vitalsign_max'
]

# 4. Chemistry (Hóa sinh máu)
chem_cols = [
    'creatinine_min', 'bun_max', 
    'bicarbonate_min', 
    'potassium_chemistry_max', 'sodium_chemistry_median', 
    'aniongap_max', 'albumin_median', 
    'glucose_chemistry_max'
]

# 5. Urine Output
uo_cols = ['urineoutput_12h_sum', 'uo_rate_ml_kg_hr']

# 6. CBC (Công thức máu)
cbc_cols = [
    'hemoglobin_min', 'hematocrit_min', 'wbc_max', 
    'platelet_median', 'rdw_median'
]

# 7. Blood Differential
diff_cols = ['nlr_max', 'bands_max']

# 8. Cardiac Markers
cardiac_cols = [
    'troponin_t_max', 'ck_mb_max', 'ntprobnp_max'
]

# 9. Coagulation
coag_cols = [
    'inr_median', 'pt_median', 'ptt_median', 
    'fibrinogen_median', 'd_dimer_median'
]

# 10. Enzymes
enzyme_cols = [
    'ck_cpk_max', 'ld_ldh_max', 
    'alt_max', 'ast_max', 'bilirubin_total_max', 
    'alp_median', 'ggt_median', 'amylase_median'
]

# 11. Scores & Rhythm
score_cols = [
    'gcs_min', 'gcs_motor_min', 'gcs_unable_max',
    'is_afib', 'is_vt_vf', 'is_paced', 'is_high_grade_block'
]

# 12. Interventions & Medications
treatment_cols = [
    # Ventilation
    'invasive_vent_flag', 'non_invasive_vent_flag',
    'peep_max', 'fio2_max', 'plateau_pressure_max', 'tidal_volume_observed_max',
    
    # Vasopressor
    'vaso_flag', 'norepinephrine_equivalent_dose_max',
    
    # Infection
    'infection_suspected_flag', 'blood_culture_flag',
    
    # Medications
    'use_aspirin', 'has_acei',
    'use_vancomycin_iv', 'cnt_systemic_antibiotics',
    
    # Invasive Lines
    'has_mcs_device', 'has_pa_catheter', 'is_femoral_access', 'cnt_invasive_lines'
]

# 13. Gom tất cả lại theo thứ tự
ordered_cols = (id_cols + demo_cols + vital_cols + chem_cols + uo_cols + 
                cbc_cols + diff_cols + cardiac_cols + coag_cols + enzyme_cols + 
                score_cols + treatment_cols)

# 14. Chỉ lấy những cột thực sự tồn tại trong danh sách cols hiện tại
# (Đề phòng trường hợp code chạy thiếu một số bước tạo biến)
existing_ordered = [c for c in ordered_cols if c in cols]

# 15. Thêm những cột còn sót lại (nếu có, ví dụ los nếu chưa drop)
remaining_cols = [c for c in cols if c not in existing_ordered]
final_order = existing_ordered + remaining_cols

# 16. Sắp xếp lại DataFrame
aki_df = aki_df[final_order]

print("Tổng số cột:", len(aki_df.columns))
print("Các cột còn sót lại (nếu có):", remaining_cols)
aki_df.head()

Tổng số cột: 84
Các cột còn sót lại (nếu có): ['los']


,subject_id,hadm_id,stay_id,aki_label,gender,admission_age,race,weight,height,intime,...,blood_culture_flag,use_aspirin,has_acei,use_vancomycin_iv,cnt_systemic_antibiotics,has_mcs_device,has_pa_catheter,is_femoral_access,cnt_invasive_lines,los
0,10002155,23822395,33685454,1,F,81,WHITE,53.0,NaN,2129-08-04 12:45:00,...,0.0,1,1,0,0,0,0,1,2,6.178912
1,10002495,24982426,36753294,1,M,81,UNKNOWN,64.1,170.0,2141-05-22 20:18:01,...,0.0,1,0,0,3,0,0,0,0,5.087512
2,10002667,23197839,31573075,1,F,58,WHITE,87.7,165.0,2187-02-23 16:02:25,...,0.0,0,0,0,0,1,0,1,2,1.951921
3,10005817,20626031,32604416,1,M,66,WHITE,91.0,173.0,2132-12-15 09:29:01,...,0.0,1,0,1,1,0,1,0,3,2.359097
4,10007058,22954658,32506122,0,M,48,WHITE,79.3,NaN,2167-11-07 20:22:00,...,0.0,1,0,0,0,0,0,0,0,2.022963


In [45]:
aki_df.drop(['los', 'intime', 'outtime', 'prediction_start_time', 'prediction_end_time', 'last_careunit', 'hospstay_seq', 'uo_rate_ml_kg_hr', 'spo2_min', 'sodium_chemistry_median', 'is_vt_vf', 'd_dimer_median', 'ggt_median', 'gcs_min', 'gcs_motor_min', 'gcs_unable_max', "has_mcs_device", "infection_suspected_flag", "invasive_vent_flag", "is_femoral_access", "is_paced", "vaso_flag", "norepinephrine_equivalent_dose_max", "blood_culture_flag", "has_acei", "use_vancomycin_iv", "cnt_systemic_antibiotics", "has_pa_catheter", "cnt_invasive_lines"], axis=1, inplace=True)
# aki_df.drop(['los', 'intime', 'outtime', 'prediction_start_time', 'prediction_end_time', 'last_careunit', 'hospstay_seq', 'gcs_min', 'gcs_motor_min'], axis=1, inplace=True)
aki_df.columns.tolist()

/tmp/ipykernel_186264/755142629.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aki_df.drop(['los', 'intime', 'outtime', 'prediction_start_time', 'prediction_end_time', 'last_careunit', 'hospstay_seq', 'uo_rate_ml_kg_hr', 'spo2_min', 'sodium_chemistry_median', 'is_vt_vf', 'd_dimer_median', 'ggt_median', 'gcs_min', 'gcs_motor_min', 'gcs_unable_max', "has_mcs_device", "infection_suspected_flag", "invasive_vent_flag", "is_femoral_access", "is_paced", "vaso_flag", "norepinephrine_equivalent_dose_max", "blood_culture_flag", "has_acei", "use_vancomycin_iv", "cnt_systemic_antibiotics", "has_pa_catheter", "cnt_invasive_lines"], axis=1, inplace=True)


['subject_id',
 'hadm_id',
 'stay_id',
 'aki_label',
 'gender',
 'admission_age',
 'race',
 'weight',
 'height',
 'first_careunit',
 'first_hosp_stay',
 'heart_rate_max',
 'sys_bp_min',
 'mean_bp_min',
 'dias_bp_min',
 'resp_rate_max',
 'temperature_max',
 'glucose_vitalsign_max',
 'creatinine_min',
 'bun_max',
 'bicarbonate_min',
 'potassium_chemistry_max',
 'aniongap_max',
 'albumin_median',
 'glucose_chemistry_max',
 'urineoutput_12h_sum',
 'hemoglobin_min',
 'hematocrit_min',
 'wbc_max',
 'platelet_median',
 'rdw_median',
 'nlr_max',
 'bands_max',
 'troponin_t_max',
 'ck_mb_max',
 'ntprobnp_max',
 'inr_median',
 'pt_median',
 'ptt_median',
 'fibrinogen_median',
 'ck_cpk_max',
 'ld_ldh_max',
 'alt_max',
 'ast_max',
 'bilirubin_total_max',
 'alp_median',
 'amylase_median',
 'is_afib',
 'is_high_grade_block',
 'non_invasive_vent_flag',
 'peep_max',
 'fio2_max',
 'plateau_pressure_max',
 'tidal_volume_observed_max',
 'use_aspirin']

In [46]:
aki_df.to_csv('/media/data3/users/tpp/MIMICIV_AKI_AMI/Thai/AKI_patients_with_features.csv')

In [47]:
aki_df

,subject_id,hadm_id,stay_id,aki_label,gender,admission_age,race,weight,height,first_careunit,...,alp_median,amylase_median,is_afib,is_high_grade_block,non_invasive_vent_flag,peep_max,fio2_max,plateau_pressure_max,tidal_volume_observed_max,use_aspirin
0,10002155,23822395,33685454,1,F,81,WHITE,53.0,NaN,Coronary Care Unit (CCU),...,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,1
1,10002495,24982426,36753294,1,M,81,UNKNOWN,64.1,170.0,Coronary Care Unit (CCU),...,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,1
2,10002667,23197839,31573075,1,F,58,WHITE,87.7,165.0,Cardiac Vascular Intensive Care Unit (CVICU),...,97.0,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0
3,10005817,20626031,32604416,1,M,66,WHITE,91.0,173.0,Cardiac Vascular Intensive Care Unit (CVICU),...,NaN,NaN,0.0,0.0,0.0,5.0,100.0,NaN,410.0,1
4,10007058,22954658,32506122,0,M,48,WHITE,79.3,NaN,Cardiac Vascular Intensive Care Unit (CVICU),...,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2311,19981210,26790133,38232926,1,M,78,WHITE,74.0,157.0,Coronary Care Unit (CCU),...,NaN,NaN,1.0,0.0,0.0,NaN,NaN,NaN,NaN,1
2312,19986880,22950392,37092788,1,F,78,UNKNOWN,58.8,NaN,Coronary Care Unit (CCU),...,NaN,NaN,0.0,1.0,0.0,NaN,NaN,NaN,NaN,1
2313,19989437,25692702,34737716,1,M,38,WHITE,113.9,175.0,Cardiac Vascular Intensive Care Unit (CVICU),...,NaN,NaN,0.0,0.0,0.0,8.0,100.0,NaN,587.0,0
2314,19991743,20346977,36005469,1,F,84,WHITE,53.0,NaN,Coronary Care Unit (CCU),...,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,1
